# Compare Integrated CSV vs Parquet

This notebook loads the integrated CSV and Parquet files from `raw_files`, then checks whether they contain the same data after normalizing common format differences such as column order, row order, date parsing, and numeric precision.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RAW_FILES_DIR = Path("raw_files")
CSV_PATH = RAW_FILES_DIR / "integrated_monthly_base.csv"
PARQUET_PATH = RAW_FILES_DIR / "integrated_monthly_base.parquet"

CSV_PATH, PARQUET_PATH

## Load Files

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")
if not PARQUET_PATH.exists():
    raise FileNotFoundError(f"Parquet file not found: {PARQUET_PATH}")

csv_df = pd.read_csv(CSV_PATH)
parquet_df = pd.read_parquet(PARQUET_PATH)

print(f"CSV source:     {CSV_PATH}")
print(f"Parquet source: {PARQUET_PATH}")
print(f"CSV shape:      {csv_df.shape}")
print(f"Parquet shape:  {parquet_df.shape}")

## Quick Inspection

In [ ]:
display(csv_df.head())
display(parquet_df.head())

summary = pd.DataFrame(
    {
        "csv_dtype": csv_df.dtypes.astype(str),
        "parquet_dtype": parquet_df.dtypes.astype(str),
        "csv_nulls": csv_df.isna().sum(),
        "parquet_nulls": parquet_df.isna().sum(),
    }
)
display(summary)

## Normalize for Fair Comparison

In [ ]:
def canonicalize_for_compare(df: pd.DataFrame) -> pd.DataFrame:
    working = df.copy()
    working.columns = [str(c).strip() for c in working.columns]

    for col in working.columns:
        col_lower = col.lower()
        if col_lower == "date" or col_lower.endswith("_at") or col_lower == "period_end":
            parsed = pd.to_datetime(working[col], errors="coerce")
            if parsed.notna().any():
                working[col] = parsed.dt.tz_localize(None)

    for col in working.select_dtypes(include="object").columns:
        numeric = pd.to_numeric(working[col], errors="coerce")
        original_non_null = working[col].notna().sum()
        numeric_non_null = numeric.notna().sum()
        if original_non_null > 0 and numeric_non_null == original_non_null:
            working[col] = numeric
        else:
            working[col] = working[col].astype("string").str.strip()

    ordered_cols = sorted(working.columns)
    working = working[ordered_cols]

    sort_cols = ["date"] if "date" in working.columns else ordered_cols
    working = working.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
    return working


csv_cmp = canonicalize_for_compare(csv_df)
parquet_cmp = canonicalize_for_compare(parquet_df)

print(f"Canonical CSV shape:     {csv_cmp.shape}")
print(f"Canonical Parquet shape: {parquet_cmp.shape}")

## Compare

In [ ]:
def compare_frames(left: pd.DataFrame, right: pd.DataFrame, *, rtol: float = 1e-9, atol: float = 1e-9) -> dict:
    results = {
        "same_shape": left.shape == right.shape,
        "same_columns": list(left.columns) == list(right.columns),
        "missing_from_csv": sorted(set(right.columns) - set(left.columns)),
        "missing_from_parquet": sorted(set(left.columns) - set(right.columns)),
        "equal": False,
        "message": "",
    }

    common_cols = sorted(set(left.columns) & set(right.columns))
    left_common = left[common_cols].copy()
    right_common = right[common_cols].copy()

    try:
        assert_frame_equal(
            left_common,
            right_common,
            check_dtype=False,
            check_like=True,
            check_exact=False,
            rtol=rtol,
            atol=atol,
        )
        results["equal"] = results["same_shape"] and results["same_columns"]
        results["message"] = "Common columns match within tolerance."
    except AssertionError as exc:
        results["message"] = str(exc)

    return results


comparison = compare_frames(csv_cmp, parquet_cmp)
comparison

## Difference Details

Run this cell if the comparison above is not fully equal. It reports row/column-level differences for the shared columns.

In [ ]:
def cell_values_equal(left_value, right_value, *, rtol: float = 1e-9, atol: float = 1e-9) -> bool:
    left_missing = pd.isna(left_value)
    right_missing = pd.isna(right_value)
    if left_missing or right_missing:
        return bool(left_missing and right_missing)

    if isinstance(left_value, (int, float, np.number)) and isinstance(right_value, (int, float, np.number)):
        return bool(np.isclose(left_value, right_value, rtol=rtol, atol=atol, equal_nan=True))

    return bool(left_value == right_value)


def build_difference_table(left: pd.DataFrame, right: pd.DataFrame, max_diffs: int = 200) -> pd.DataFrame:
    common_cols = sorted(set(left.columns) & set(right.columns))
    max_rows = min(len(left), len(right))
    diffs = []

    for row_idx in range(max_rows):
        for col in common_cols:
            left_value = left.iloc[row_idx][col]
            right_value = right.iloc[row_idx][col]
            if not cell_values_equal(left_value, right_value):
                diffs.append(
                    {
                        "row": row_idx,
                        "column": col,
                        "csv_value": left_value,
                        "parquet_value": right_value,
                    }
                )
                if len(diffs) >= max_diffs:
                    return pd.DataFrame(diffs)

    return pd.DataFrame(diffs)


print(comparison["message"])

if comparison["missing_from_csv"]:
    print("Columns missing from CSV:", comparison["missing_from_csv"])
if comparison["missing_from_parquet"]:
    print("Columns missing from Parquet:", comparison["missing_from_parquet"])

diff_table = build_difference_table(csv_cmp, parquet_cmp)
display(diff_table)

if len(csv_cmp) != len(parquet_cmp):
    print(f"Row count differs: CSV={len(csv_cmp)}, Parquet={len(parquet_cmp)}")

## Final Verdict

In [ ]:
if comparison["equal"]:
    print("PASS: The CSV and Parquet data are the same after canonicalization.")
else:
    print("REVIEW: The CSV and Parquet data are not fully the same after canonicalization.")
    print("Check the comparison dictionary and difference table above.")